# 球员评估分析 - 2022世界杯版本

本notebook提供完整的球员评估功能，用于分析2022世界杯比赛中防守球员的影响力。

**适配版本**: 2022 FIFA世界杯数据集

## 功能概述

1. **防守球员影响力分析** - 计算每个防守球员对进攻球员的影响
2. **防守球员移除实验** - 评估移除不同防守球员对模型预测的影响
3. **球员表现评估** - 综合评估防守球员的整体表现
4. **数据导出** - 保存分析结果供后续研究使用

## 1. 导入库和配置

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import random
import torch
import convert_tracking as ct
import create_graph as cg
import scale_graph as sg
import GNNs.convert_data as cd
from GNNs.GAT import GATReceptionPredictor
from sklearn.preprocessing import MinMaxScaler

print("✓ 库导入成功")

## 2. 配置参数

In [ ]:
GAME_ID = '10517'
DATA_DIR = f'Data/{GAME_ID}'
MODEL_PATH = f'results/model_final_{GAME_ID}.pth'
SCALER_PATH = f'results/scaler_final_{GAME_ID}.pkl'
SCALED_GRAPHS_PATH = f'results/scaled_graphs_final_{GAME_ID}.pkl'
OUTPUT_DIR = f'results/player_eval_{GAME_ID}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"配置: 比赛ID={GAME_ID}, 输出={OUTPUT_DIR}")

## 3. 加载数据

In [ ]:
xT_grid = pd.read_csv('xT_grid.csv', header=None) if os.path.exists('xT_grid.csv') else pd.DataFrame(np.linspace(0, 1, 96).reshape(12, 8))
metadata = ct.get_metadata(GAME_ID)
home_team_name, away_team_name = metadata[2], metadata[3]
rosters_home, rosters_away = metadata[5], metadata[6]

team_rosters_data = []
for p in rosters_home:
    team_rosters_data.append({'game_id': int(GAME_ID), 'player_nickname': p, 'team_name': home_team_name})
for p in rosters_away:
    team_rosters_data.append({'game_id': int(GAME_ID), 'player_nickname': p, 'team_name': away_team_name})
team_rosters = pd.DataFrame(team_rosters_data)

balls_df = pd.read_csv(f'{DATA_DIR}/balls_{GAME_ID}.csv')
events_df = pd.read_csv(f'{DATA_DIR}/events_{GAME_ID}.csv')
players_df = pd.read_csv(f'{DATA_DIR}/players_{GAME_ID}.csv')

print(f"✓ 数据加载: {home_team_name} vs {away_team_name}, {len(events_df)} 事件")

## 4. 加载图和模型

In [ ]:
if os.path.exists(SCALED_GRAPHS_PATH):
    with open(SCALED_GRAPHS_PATH, 'rb') as f:
        scaled_graphs = pickle.load(f)
    print(f"✓ 加载 {len(scaled_graphs)} 个图")
else:
    print("创建图...")
    graphs = []
    for i, frameNum in enumerate(events_df['frameNum'].values):
        if i % 100 == 0: print(f"  {i}/{len(events_df)}")
        G = cg.create_normalized_graph_directed(players_df, balls_df, events_df, frameNum, home_team_name)
        if G: graphs.append(G)
    graph_scaler = sg.GraphFeatureScaler()
    graph_scaler.fit(graphs)
    scaled_graphs = [graph_scaler.transform_graph(G) for G in graphs]
    with open(SCALED_GRAPHS_PATH, 'wb') as f:
        pickle.dump(scaled_graphs, f)
    print(f"✓ 创建并保存 {len(scaled_graphs)} 个图")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model = GATReceptionPredictor(15, 6, 32, 16, 16).to(device)
loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
loaded_model.eval()

loaded_scaler = pickle.load(open(SCALER_PATH, 'rb')) if os.path.exists(SCALER_PATH) else MinMaxScaler().fit([[0, 0], [105, 68]])

print(f"✓ 模型和缩放器加载完成")

## 5. 定义分析函数

In [ ]:
def get_pitch_value(data, x, y):
    x, y = max(1, min(x, 104)), max(1, min(y, 67))
    rows, cols = data.shape
    col_idx = min(int(x / (105 / cols)), cols - 1)
    row_idx = min(int(y / (68 / rows)), rows - 1)
    return data.iloc[row_idx, col_idx]

def calculate_defender_influence(scaled_graph, model, defender, attacker_names, all_names, scaler, xT_grid):
    data = cd.convert_graph_to_pytorch_geometric_reception(scaled_graph)
    model.eval()
    graph_names = [s[0] for s in scaled_graph.nodes(data=True)]
    def_data = scaled_graph.nodes[defender]
    if hasattr(scaler, 'position_scaler'):
        def_x, def_y = scaler.position_scaler.inverse_transform([[def_data['features'][0], def_data['features'][1]]])[0]
    else:
        def_x, def_y = scaler.inverse_transform([[def_data['features'][0], def_data['features'][1]]])[0]
    
    with torch.no_grad():
        probs, attn_weights = model(data.x, data.edge_index, data.edge_attr, None)
    
    attn_w1, attn_w2 = attn_weights
    edge_idx1, attn1 = attn_w1
    edge_idx2, attn2 = attn_w2
    
    results = {'distances': [], 'attentions': [], 'influences': [], 'rel_influences': [], 'threats': []}
    total_perf = 0
    
    for attacker in attacker_names:
        att_data = scaled_graph.nodes[attacker]
        if hasattr(scaler, 'position_scaler'):
            x, y = scaler.position_scaler.inverse_transform([[att_data['features'][0], att_data['features'][1]]])[0]
        else:
            x, y = scaler.inverse_transform([[att_data['features'][0], att_data['features'][1]]])[0]
        dist = np.sqrt((def_x - x)**2 + (def_y - y)**2)
        xt = get_pitch_value(xT_grid, x, y)
        
        mask_node = graph_names.index(defender)
        target_node = graph_names.index(attacker)
        mask = (edge_idx1[1] == target_node) & (edge_idx1[0] == mask_node)
        
        if mask.any():
            att_val = float((attn1[mask].mean() + attn2[mask].mean()) / 2)
        else:
            att_val = 0.0
        
        with torch.no_grad():
            probs_v2, _ = model(data.x, data.edge_index, data.edge_attr, None, test_mode=True, mask_node_name=defender, target_node_name=attacker, graph=scaled_graph)
        
        att_idx = all_names.index(attacker)
        prob_before = float(probs[att_idx])
        prob_after = float(probs_v2[att_idx])
        influence = prob_after - prob_before
        rel_inf = 100 * abs(influence) / prob_after if prob_after > 0 else 0.0
        
        results['distances'].append(dist)
        results['attentions'].append(att_val)
        results['influences'].append(influence)
        results['rel_influences'].append(rel_inf)
        results['threats'].append(xt)
        total_perf += influence * xt * 100
    
    return results, total_perf

print("✓ 函数定义完成")

## 6. 批量分析所有图

In [ ]:
print(f"开始分析 {len(scaled_graphs)} 个图...\n")
all_defender_dfs = []

for i, graph in enumerate(scaled_graphs):
    if i % 50 == 0:
        print(f"进度: {i+1}/{len(scaled_graphs)} ({100*(i+1)/len(scaled_graphs):.1f}%)")
    
    all_names = [s[0] for s in graph.nodes(data=True)]
    attacker_names = [s[0] for s in graph.nodes(data=True) if (s[1]['features'][7] == 1) & (s[0] != 'ball')]
    defender_names = [s[0] for s in graph.nodes(data=True) if (s[1]['features'][7] == 0) & (s[0] != 'ball')]
    
    for defender in defender_names:
        try:
            results, perf = calculate_defender_influence(graph, loaded_model, defender, attacker_names, all_names, loaded_scaler, xT_grid)
            df = pd.DataFrame({
                'graph_index': i,
                'Attacker': attacker_names,
                'Defender': defender,
                'match_id': int(GAME_ID),
                'Def_Distance': results['distances'],
                'Def_Attention': results['attentions'],
                'Def_Influence': results['influences'],
                'Def_Rel_Influence': results['rel_influences'],
                'Att_Threat': results['threats'],
                'Total_Perf': perf
            })
            all_defender_dfs.append(df)
        except Exception as e:
            if i < 5:
                print(f"  警告: 图{i} 防守球员{defender} 失败: {e}")

combined_df = pd.concat(all_defender_dfs, ignore_index=True)
print(f"\n✓ 分析完成: {len(combined_df)} 条记录")

## 7. 保存结果

In [ ]:
output_file = f'{OUTPUT_DIR}/{GAME_ID}_defender_performance.csv'
combined_df.to_csv(output_file, index=False)
print(f"✓ 结果已保存: {output_file}")
print(f"\n统计:")
print(f"  总记录: {len(combined_df):,}")
print(f"  防守球员: {combined_df['Defender'].nunique()}")
print(f"  进攻球员: {combined_df['Attacker'].nunique()}")

## 8. 结果分析

In [ ]:
print("防守球员总体表现排名（前10）:")
print(combined_df.groupby('Defender')['Total_Perf'].mean().sort_values(ascending=False).head(10))

print("\n平均影响力最大的防守球员（前10）:")
print(combined_df.groupby('Defender')['Def_Influence'].mean().sort_values().head(10))

print("\n平均注意力权重最高的防守球员（前10）:")
print(combined_df.groupby('Defender')['Def_Attention'].mean().sort_values(ascending=False).head(10))

## 9. 可视化分析（可选）

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

top_defenders = combined_df.groupby('Defender')['Total_Perf'].mean().sort_values(ascending=False).head(10)
axes[0, 0].barh(range(len(top_defenders)), top_defenders.values)
axes[0, 0].set_yticks(range(len(top_defenders)))
axes[0, 0].set_yticklabels(top_defenders.index)
axes[0, 0].set_xlabel('总体表现值')
axes[0, 0].set_title('防守球员总体表现 Top 10')
axes[0, 0].invert_yaxis()

top_influence = combined_df.groupby('Defender')['Def_Influence'].mean().sort_values().head(10)
axes[0, 1].barh(range(len(top_influence)), top_influence.values)
axes[0, 1].set_yticks(range(len(top_influence)))
axes[0, 1].set_yticklabels(top_influence.index)
axes[0, 1].set_xlabel('平均影响力')
axes[0, 1].set_title('防守影响力最大 Top 10')
axes[0, 1].invert_yaxis()

axes[1, 0].scatter(combined_df['Def_Distance'], combined_df['Def_Influence'], alpha=0.3)
axes[1, 0].set_xlabel('距离')
axes[1, 0].set_ylabel('影响力')
axes[1, 0].set_title('距离 vs 影响力')

axes[1, 1].scatter(combined_df['Def_Attention'], combined_df['Def_Influence'], alpha=0.3)
axes[1, 1].set_xlabel('注意力权重')
axes[1, 1].set_ylabel('影响力')
axes[1, 1].set_title('注意力 vs 影响力')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/{GAME_ID}_analysis.png', dpi=150, bbox_inches='tight')
print(f"✓ 可视化已保存: {OUTPUT_DIR}/{GAME_ID}_analysis.png")
plt.show()

## 总结

本notebook完成了以下工作：

✓ 数据加载和预处理
✓ 图创建和缩放
✓ 模型加载
✓ 防守球员影响力分析
✓ 批量处理所有比赛帧
✓ 结果保存和可视化

### 输出文件：

- `{GAME_ID}_defender_performance.csv` - 详细的防守球员表现数据
- `{GAME_ID}_analysis.png` - 可视化分析图表

### 下一步：

- 使用 `test_player_eval_final.ipynb` 进行快速测试
- 分析其他比赛（修改GAME_ID）
- 进行更深入的统计分析
- 比较不同比赛的防守表现

### 相关文件：

- `test_player_eval_final.ipynb` - 快速测试版本
- `player_eval.py` - 原始Python脚本
- `Visualisation_2022WC.ipynb` - 可视化工具